In [17]:
import os
import json
from datetime import datetime, timezone

In [31]:
def year_to_milliseconds(year):
    # Create a datetime object, the format is UTC, is not the same as local.
    start_of_year = datetime(year, 1, 1, 0, 0, 0, tzinfo=timezone.utc) #year, month, day, hour, minute
    
    # Convert the datetime object to milliseconds since Epoch
    milliseconds_since_epoch = int(start_of_year.timestamp() * 1000)
    
    return milliseconds_since_epoch

In [32]:
def get_file_list(root_folder, firstlayer, number_resources):
    """This script will return a list of the files, it searched for the first one and take the next ones depending on the range of the years

    Args:
        root_folder (_type_): _description_
        updates (_type_): _description_

    Returns:
        _type_: _description_
    """
    file_list = []
    append_files = False  # Flag to indicate when to start appending file paths
    folders_processed = 0

    # Iterate over all folders within the main folder
    for folder_name in os.listdir(root_folder):
        folder_path = os.path.join(root_folder, folder_name) # build the folder path
        file_path = os.path.join(folder_path, "resource.json")
        # Check if the folder is the "localName" one
        if folder_name == firstlayer:
            file_list.append(file_path)
            append_files = True  # Set the flag to start appending files
            continue  # Move to the next iteration without appending "resource.json" in this folder

        # Append file paths if the flag is set and less than the amount of number_resources
        if append_files and folders_processed < number_resources - 1: # updates["number_resources"]
            file_list.append(file_path)
            folders_processed += 1  # Increment the counter
    
    return file_list

In [36]:
def update_json_files(file_list, year_list, metadata_update):
    """
    Update JSON files in folders named "first_resource_name" within the specified year range.

    Args:
    - root_folder (str): Path to the root folder containing folders named "first_resource_name".
    - updates (dict): Dictionary containing the updates to be applied to the JSON files.
    """
    # they should go in pairs
    for year, file_path in zip(year_list, file_list):
        with open(file_path, 'r') as file:
            try:
                # Load JSON data into a dict
                data = json.load(file)
            except json.JSONDecodeError as e:
                # Handle JSON decoding errors
                print(f"Error decoding JSON file {file_path}: {e}")
                continue

        # Build the time string
        tstart = year_to_milliseconds(year)
        tend = year_to_milliseconds(year + 1)
        time_string = "τ1{{tend={},tstart={},ttype=logical}}".format(tend,tstart) # This is the structure: τ1{tend=962409600000,tstart=646790400000,ttype=logical}
        
        # Open the geometry string and update it
        geometry_string = data["geometry"]
        # Find the index of the first occurrence of "S2"
        index_of_s2 = geometry_string.find("S2")
        # build the string
        updated_geometry_string = time_string + geometry_string[index_of_s2:]
        # update the field
        data["geometry"] = updated_geometry_string
        
        # Update the rest of the dict with the update values
        data["metadata"] = metadata_update
        
        """Uncoment this part once you are sure what you are doing"""
        # Open the JSON file for writing
        # with open(file_path, 'w') as file:
        #     # Write the updated JSON data back to the file
        #     json.dump(data, file, indent=4)
        #     # Print a message indicating that the file has been updated
        #     print(f"Updated JSON file: {file_path}")
    return data


In [37]:
root_folder = r"C:\Users\ruben.crespo\git\aries.heco\resources"
firstlayer = "im-data-global-conservation.ideam_forest_change_coverage_colombia_1990_2000_v5"

"""These two have to match"""
number_resources = 10
year_list = range(2000, 2010 + 1, 1) # or give a set of unique years [2001, 2002]


"""Define the content""" # - \r\n = Enter
Title = "Change in area covered by natural forest from year 1990 to 2000"
Description = "Periodic monitoring of the change in the area covered by natural forest allows quantifying the difference or net balance between the area of ​​regenerated forest (gain) and the area of ​​deforested forest (loss) that occur in an analyzed period of time. Likewise, the areas that remain unchanged in a period of time analyzed are identified, either as stable forest areas or as non-stable forest areas. This continuous monitoring provides information for decision-making and control actions in the areas of greatest deforestation in Colombia. Natural forest area change data is generated from semi-automated digital processing of medium-resolution remote sensing images. This monitoring has been carried out annually at the national level since 2013 and additionally, there is historical information for the periods 1990-2000, 2000-2005, 2005-2010 and 2010-2012. The results obtained are presented at the national and departmental levels and in the jurisdiction of the regional autonomous corporations. For the processing and reporting of changes in forest cover, the technical guidelines defined in the Digital Image Processing Protocol for the quantification of deforestation in Colombia Version 2.0 (Galindo et al., IDEAM 2014) are followed, in which identification of deforestation or regeneration is done through direct detection of changes. In this way, the satellite images of the two analyzed dates are simultaneously processed and compared, and variations in the spectral response that may correspond to a loss or gain of forest cover are identified. This allows direct comparison of the images and not the forest area maps generated independently for each date. Deforestation Rate The annual deforestation rate is the variation of the area covered by forest, between the initial year and the final year of the analysis period. The SMByC has generated this indicator on an annual basis since 2013 and additionally has historical information for the periods 1990-2000, 2000-2005, 2005-2010 and 2010-2012. The results obtained are presented at the national and departmental levels and in the jurisdiction of the regional autonomous corporations."
Originating_institution = "IDEAM"
Url_doi = "http://www.siac.gov.co/catalogo-de-mapas"
Autors = "IDEAM autors"
Theme = "Conservation"
Geographical_region = "Global"
Keywords = "Columbia, forest change"
References = "Protocolo de procesamiento digital de imágenes para la cuantificación de la deforestación en Colombia Versión 2.0 (Galindo et al., IDEAM 2014)\r\n"
Notes = "1. Temporal boundaries of the time series data are annotated as starting from 1st July of the year and ending till 30th June the next year.\r\n2. Values are classified into: 1-Stable forest, 2-Deforestation, 3-No information, 4-Regeneration, 5-Not stable forest. "

metadata_update = {
    "im:keywords" : Keywords,
    "dc:comment" : Description,
    "im:notes" : Notes,
    "dc:title" : Title,
    "dc:url" : Url_doi,
    "dc:creator" : Autors,
    "im:thematic-area" : Theme,
    "dc:originator" : Originating_institution,
    "im:geographic-area" : Geographical_region,
    "dc:source" : References
    }
file_list = get_file_list(root_folder, firstlayer, number_resources)
# Make sure you are changing what you want
print(file_list[:])



{'urn': 'local:rubencc:aries.heco:im-data-global-conservation.ideam_forest_change_coverage_colombia_1990_2000_v5', 'version': '0.0.1', 'adapterType': 'wcs', 'localPath': 'aries.heco/resources/im-data-global-conservation.ideam_forest_change_coverage_colombia_1990_2000_v5', 'geometry': 'τ1{tend=978307200000,tstart=946684800000,ttype=logical}S2(45205,61617){bbox=[-79.10220581116046 -66.65601461549821 -4.244791037448113 12.478016692297203],proj=EPSG:4326}', 'projectName': 'aries.heco', 'localName': 'im-data-global-conservation.ideam_forest_change_coverage_colombia_1990_2000_v5', 'type': 'NUMBER', 'resourceTimestamp': 1647028617247, 'metadata': {'im:keywords': 'Columbia, forest change', 'dc:comment': 'Periodic monitoring of the change in the area covered by natural forest allows quantifying the difference or net balance between the area of \u200b\u200bregenerated forest (gain) and the area of \u200b\u200bdeforested forest (loss) that occur in an analyzed period of time. Likewise, the areas 

In [ ]:
"""PENDING: ad year if we want to append it into any of the texts"""

data_1 = update_json_files(file_list[0:1], year_list[0:1], metadata_update) # for testing
print(data_1)